# Fase 3: Arquitectura del Vision Transformer (ViT), Dataloader Defensivo y Estudio de Ablación.

### 🎯 Objetivo de la Fase 3

Construir el codificador Vision Transformer (ViT), entrenarlo estrictamente sobre las clases de conjunto cerrado (Closed-Set) manejando el desbalanceo poblacional de forma nativa, y ejecutar el estudio de ablación iterativo para descubrir empíricamente la ventana óptima de paquetes ($N_{min}$).

### 📋 Requerimientos que debemos cumplir (Según el SRS)

1. **FR4.1 (Aplicación Dinámica del Escalado):** El Dataloader debe leer el JSON de la Fase 2 y aplicar la estandarización $[0, 1]$ *al vuelo* en la memoria RAM justo antes de enviar el tensor a la GPU.
2. **FR6 (Arquitectura ViT con retención OSR/XAI):** El modelo no puede ser una "caja negra" empaquetada. Debemos programarlo para que el método `forward()` devuelva tres cosas: los *logits* (para clasificación), el token `[CLS]` (para la distancia de Mahalanobis en la Fase 4) y los *Attention Weights* (para la Interpretabilidad en la Fase 5).
3. **FR7 (Desbalanceo sin Interpolación):** Queda estrictamente prohibido usar SMOTE/ADASYN. El desbalanceo se mitigará inyectando `Class Weights` directamente en la función `CrossEntropyLoss`.
4. **NFR5 (Eficiencia AMP):** Obligatorio el uso de Precisión Mixta Automática (AMP) de PyTorch (`torch.cuda.amp.autocast`) para comprimir los tensores espaciales en la VRAM y maximizar el `batch_size`.
5. **FR11 (Persistencia de Estado / Checkpointing):** Guardado automático del estado (pesos, optimizador y RNG) para evitar la pérdida de progreso si la VM de GCP se apaga.

---

### 🛠️ Pasos para la Implementación (Orden de Desarrollo)

Para mantener la modularidad, dividiremos esta fase en tres componentes clave que vivirán en la carpeta `src/models/` y `src/data_ingestion/`.

#### Paso 1: El "Dataloader Defensivo" (El Puente I/O)

* **Lógica:** Crear una clase `Dataset` en PyTorch que lea los HDF5 en modo SWMR (`swmr=True`).
* **Truncamiento Dinámico:** El Dataloader recibirá el hiperparámetro $N_{min}$. Si el tensor HDF5 tiene 18 paquetes pero el experimento actual evalúa $N_{min}=6$, el Dataloader recortará el tensor en RAM antes de pasarlo.
* **Ensamblaje RGB-E:** Tomará los `raw_bytes` (Rojo/Verde) y la `blue_channel_entropy` (Azul), aplicará la normalización usando los valores de `scaler_bounds.json`, y construirá el tensor tridimensional final `(Canales, N_min, Longitud_Maxima)`.
* **Rendimiento:** El `DataLoader` final se instanciará con `pin_memory=True` y `num_workers > 4` para mantener la GPU NVIDIA L4 alimentada sin inanición (*starvation*).

#### Paso 2: La Arquitectura del Vision Transformer (ViT Base)

* **Patch Embedding:** Usaremos la librería `einops` para dividir limpiamente el tensor tridimensional en parches uniformes.
* **Positional Encoding & [CLS]:** Inyectaremos el parámetro aprendible `[CLS]` al inicio de la secuencia y sumaremos la codificación posicional para mantener la noción temporal del flujo.
* **Transformer Encoder:** Implementaremos bloques de Autoatención Multicabezal (MHSA).
* **El Cabezal (MLP Head):** Una capa lineal final que mapeará la representación a las clases conocidas (Benign, DoS, Web, etc.). **Ojo:** No aplicaremos `Softmax` internamente, devolveremos los *logits* crudos para que `CrossEntropyLoss` haga el cálculo numérico estable.

#### Paso 3: El Orquestador del Estudio de Ablación

* **Bucle Externo:** Un script que itere sobre el espacio de búsqueda: `for N_min in [3, 6, 9, 12, 15, 18]:`
* **Pesos de Clase:** Calcular dinámicamente el inverso de las frecuencias de clase del conjunto de entrenamiento y pasarlo al `CrossEntropyLoss(weight=class_weights)`.
* **Entrenamiento (Bucle Interno):** Implementar el *GradScaler* para la Precisión Mixta (AMP). Entrenar por $X$ epochs.
* **Métrica Principal:** Evaluar el rendimiento en el conjunto de validación utilizando estrictamente el **Coeficiente de Correlación de Matthews (MCC)**.

#### Paso 4: Telemetría y Checkpointing

* Integrar `mlflow` o `wandb` al inicio del bucle para que cada $N_{min}$ sea un experimento separado (*run*).
* Registrar el MCC, la pérdida y la latencia del *forward pass*.
* Guardar los pesos `.pt` del modelo que obtenga el mejor MCC para cada $N_{min}$.

---

### 📦 Entradas y Salidas Esperadas

* **Entrada:** Directorio `data/processed/train_val/` y `configs/scaler_bounds.json`.
* **Salida:** Gráficas de convergencia (Loss/MCC) en tu plataforma de telemetría, y los archivos `.pt` (pesos del modelo) guardados en la carpeta `checkpoints/`.

### ✅ Criterio de Validación para avanzar a la Fase 4

La fase se considerará exitosa si:

1. La GPU (`nvidia-smi`) reporta una utilización sostenida superior al 85% durante el entrenamiento (lo que indica que el Dataloader no es un cuello de botella).
2. El modelo converge (la pérdida disminuye y el MCC supera el 0.70 en validación).
3. Se identifica un $N_{min}$ claro donde el MCC alcanza un estancamiento (*plateau*), demostrando que no necesitamos procesar el flujo completo para detectar el ataque.


In [1]:
# ==============================================================================
# FASE 2: PERFILAMIENTO MIN-MAX (ADAPTACIÓN PARA COLAB)
# ==============================================================================
import os
import h5py
import json
import numpy as np

# 1. Simulamos el YAML global para que apunte a las carpetas locales de Colab
GLOBAL_CONFIG = {
    'paths': {
        'configs': {'scaler_bounds': 'scaler_bounds.json'},
        'output': {'train_val': 'data/'} # Aquí es donde subiste los 2 archivos .hdf5
    }
}

TRAIN_DIR = GLOBAL_CONFIG['paths']['output']['train_val']
OUTPUT_JSON = GLOBAL_CONFIG['paths']['configs']['scaler_bounds']
CHECKPOINT_FILE = OUTPUT_JSON.replace(".json", "_checkpoint.json")

def calculate_global_bounds():
    print("=======================================================")
    print(" INICIANDO PERFILAMIENTO GLOBAL MIN-MAX (MODO SANDBOX)")
    print("=======================================================")

    # ESTADO BASE Y RESILIENCIA (Idempotencia)
    state = {
        "global_min_entropy": None,
        "global_max_entropy": None,
        "global_min_raw": None,
        "global_max_raw": None,
        "processed_files": []
    }

    # Cargar progreso previo si hubo una desconexión
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            saved_state = json.load(f)
            state.update(saved_state)
        print(f"[*] Rescatando sesión interrumpida. Archivos ya procesados: {len(state['processed_files'])}")

    train_files = [f for f in os.listdir(TRAIN_DIR) if f.endswith('.hdf5')]
    total_files = len(train_files)

    # Filtrar los que ya procesamos
    files_to_process = [f for f in train_files if f not in state["processed_files"]]

    if len(files_to_process) < total_files:
        print(f"[*] Omitiendo {total_files - len(files_to_process)} archivos. Procesando los {len(files_to_process)} restantes...")
    else:
        print(f"[*] Escaneando {total_files} archivos de entrenamiento en modo solo lectura...")

    for idx, filename in enumerate(files_to_process, 1):
        file_path = os.path.join(TRAIN_DIR, filename)
        try:
            with h5py.File(file_path, 'r', swmr=True) as hf:
                for flow_id in hf.keys():
                    grp = hf[flow_id]

                    # 1. Analizar Entropía (Canal Azul)
                    entropies = grp['blue_channel_entropy'][:]
                    if len(entropies) > 0:
                        min_ent = float(np.min(entropies))
                        max_ent = float(np.max(entropies))

                        if state["global_min_entropy"] is None:
                            state["global_min_entropy"] = min_ent
                            state["global_max_entropy"] = max_ent
                        else:
                            state["global_min_entropy"] = min(state["global_min_entropy"], min_ent)
                            state["global_max_entropy"] = max(state["global_max_entropy"], max_ent)

                    # 2. Analizar Bytes Crudos (Rojo/Verde) - OPTIMIZACIÓN VECTORIZADA
                    raw_ds = grp['raw_packets'][:]
                    if len(raw_ds) > 0:
                        all_bytes = np.concatenate(raw_ds) if len(raw_ds) > 1 else raw_ds[0]
                        if len(all_bytes) > 0:
                            min_raw = float(np.min(all_bytes))
                            max_raw = float(np.max(all_bytes))

                            if state["global_min_raw"] is None:
                                state["global_min_raw"] = min_raw
                                state["global_max_raw"] = max_raw
                            else:
                                state["global_min_raw"] = min(state["global_min_raw"], min_raw)
                                state["global_max_raw"] = max(state["global_max_raw"], max_raw)

            # Marcar archivo como completado
            state["processed_files"].append(filename)

            # Guardar Checkpoint Atómico
            if idx % 10 == 0 or idx == len(files_to_process):
                tmp_chk = CHECKPOINT_FILE + ".tmp"
                with open(tmp_chk, 'w') as f:
                    json.dump(state, f)
                os.rename(tmp_chk, CHECKPOINT_FILE)
                print(f"  -> Progreso guardado: {len(state['processed_files'])}/{total_files} archivos respaldados...")

        except Exception as e:
            print(f"[!] Error crítico leyendo {filename}: {str(e)}")
            continue

    # CONSOLIDACIÓN FINAL
    bounds = {
        "entropy_channel": {
            "min": state["global_min_entropy"],
            "max": state["global_max_entropy"]
        },
        "raw_bytes_channel": {
            "min": state["global_min_raw"],
            "max": state["global_max_raw"]
        }
    }

    # os.makedirs ya no es estrictamente necesario porque lo guardaremos en la raíz de Colab
    os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True) if os.path.dirname(OUTPUT_JSON) else None

    with open(OUTPUT_JSON, 'w') as f:
        json.dump(bounds, f, indent=4)

    # Limpieza: Si terminamos con éxito, borramos el checkpoint temporal
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    print("=======================================================")
    print("[✓] PERFILAMIENTO COMPLETADO Y BLINDADO.")
    print(f"[*] Entropía Min: {state['global_min_entropy']:.4f} | Max: {state['global_max_entropy']:.4f}")
    print(f"[*] Raw Bytes Min: {state['global_min_raw']:.1f} | Max: {state['global_max_raw']:.1f}")
    print(f"[*] Guardado permanentemente en: {OUTPUT_JSON}")
    print("=======================================================")

# Ejecutar directamente en la celda
calculate_global_bounds()

 INICIANDO PERFILAMIENTO GLOBAL MIN-MAX (MODO SANDBOX)
[*] Escaneando 2 archivos de entrenamiento en modo solo lectura...
  -> Progreso guardado: 2/2 archivos respaldados...
[✓] PERFILAMIENTO COMPLETADO Y BLINDADO.
[*] Entropía Min: 0.0000 | Max: 7.9955
[*] Raw Bytes Min: 0.0 | Max: 255.0
[*] Guardado permanentemente en: scaler_bounds.json


In [2]:
!pip install einops

In [6]:
# ==============================================================================
# ADAPTACIÓN PARA COLAB (Mock de Configuración)
# ==============================================================================
import os
import json
import yaml
import h5py
import torch
import logging
import argparse
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import matthews_corrcoef
from einops.layers.torch import Rearrange
from tqdm import tqdm

# Simulamos el YAML global para no tener que subirlo
GLOBAL_CONFIG = {
    'paths': {
        'configs': {'scaler_bounds': 'scaler_bounds.json'},
        'output': {'train_val': 'data/'}, # Apuntamos a la carpeta local de Colab
        'artifacts': {'checkpoints': 'checkpoints/', 'telemetry_logs': 'logs/'}
    },
    'training': {
        'epochs': 5,
        'batch_size': 16,
        'learning_rate': 0.0001
    }
}

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# ==============================================================================
# 1. EL PUENTE I/O: DATALOADER DEFENSIVO CON ESCALADO AL VUELO (FR4.1)
# ==============================================================================

def safe_collate(batch):
    # Filtrar los elementos que devolvieron None
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return torch.Tensor(), torch.Tensor() # Lote vacío de seguridad
    return torch.utils.data.dataloader.default_collate(batch)

class IDS2018Dataset(Dataset):
    def __init__(self, data_dir, scaler_json, n_min, max_bytes=128, mode='prod'):
        self.data_dir = data_dir
        self.n_min = n_min
        self.max_bytes = max_bytes
        self.mode = mode

        # Cargar fronteras matemáticas para Anti-Fuga de Datos (FR4.1)
        with open(scaler_json, 'r') as f:
            bounds = json.load(f)
            self.min_e = bounds['entropy_channel']['min']
            self.max_e = bounds['entropy_channel']['max']
            self.min_r = bounds['raw_bytes_channel']['min']
            self.max_r = bounds['raw_bytes_channel']['max']

        # Mapeo de Clases dinámico
        self.class_to_idx = {"Benign": 0, "BruteForce": 1, "DoS": 2, "DDoS": 3, "Brute_Force_Web": 4, "Brute_Force_XSS": 5, "SQL_Injection": 6}
        self.index, self.class_counts = self._build_or_load_index()

    def _build_or_load_index(self):
        index_file = os.path.join(self.data_dir, f"dataset_index_{self.mode}.pt")
        if os.path.exists(index_file):
            logging.info(f"[*] Cargando índice cacheado desde {index_file}")
            return torch.load(index_file)

        logging.info("[*] Construyendo índice maestro HDF5... (Esto tomará unos minutos la primera vez)")
        files = [f for f in os.listdir(self.data_dir) if f.endswith('.hdf5')]
        if self.mode == 'pilot': files = files[:2] # Piloto restringe a 2 archivos

        index = []
        class_counts = {v: 0 for v in self.class_to_idx.values()}

        for fname in tqdm(files, desc="Indexando"):
            path = os.path.join(self.data_dir, fname)
            try:
                with h5py.File(path, 'r', swmr=True) as hf:
                    for flow_id in hf.keys():
                        lbl_str = hf[flow_id].attrs.get('label', 'Benign')
                        if lbl_str in self.class_to_idx:
                            c_idx = self.class_to_idx[lbl_str]
                            index.append((fname, flow_id, c_idx))
                            class_counts[c_idx] += 1
            except Exception as e:
                logging.error(f"[!] Error indexando {fname}: {str(e)}")

        # Piloto estricto: Submuestreo
        if self.mode == 'pilot': index = index[:1000]

        torch.save((index, class_counts), index_file)
        return index, class_counts

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        fname, flow_id, label = self.index[idx]
        path = os.path.join(self.data_dir, fname)

        try:
            with h5py.File(path, 'r', swmr=True) as hf:
                grp = hf[flow_id]
                raw_pkts = grp['raw_packets'][:]
                entropies = grp['blue_channel_entropy'][:]

                # Truncamiento Dinámico (Estudio de Ablación)
                current_len = min(len(raw_pkts), self.n_min)

                # Ensamblaje de Tensores (Canal 0: Bytes, Canal 1: Entropía)
                img = np.zeros((2, self.n_min, self.max_bytes), dtype=np.float32)

                for i in range(current_len):
                    # Padding de bytes
                    pkt_bytes = raw_pkts[i][:self.max_bytes]
                    img[0, i, :len(pkt_bytes)] = pkt_bytes
                    img[1, i, :] = entropies[i] # Broadcast de la entropía a lo largo del paquete

                # FR4.1: Normalización Min-Max Segura
                if self.max_r > self.min_r:
                    img[0] = (img[0] - self.min_r) / (self.max_r - self.min_r)
                if self.max_e > self.min_e:
                    img[1] = (img[1] - self.min_e) / (self.max_e - self.min_e)

                return torch.tensor(img, dtype=torch.float32), torch.tensor(label, dtype=torch.long)
        except Exception as e:
            # Tolerancia a fallos: retornar matriz vacía benigna si el flujo colapsa
            logging.error(f"[!] HDF5 Read Error en {fname}, flow {flow_id}: {str(e)}")
            return None

# ==============================================================================
# 2. ARQUITECTURA: VISION TRANSFORMER OSR (FR6)
# ==============================================================================
class ViT_OSR(nn.Module):
    def __init__(self, n_min, max_bytes=128, patch_size=(1, 16), embed_dim=128, depth=4, num_heads=8, num_classes=7):
        super().__init__()
        self.patch_h, self.patch_w = patch_size
        num_patches = (n_min // self.patch_h) * (max_bytes // self.patch_w)
        patch_dim = 2 * self.patch_h * self.patch_w # 2 canales (RGB-E)

        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=self.patch_h, p2=self.patch_w),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, img):
        # Generar embeddings y token CLS
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape
        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]

        # Forward pass y extracción de pesos de atención (Simulando compatibilidad FR6)
        # Nota: PyTorch TransformerEncoder nativo no devuelve atención directamente.
        # Extraemos el token CLS post-encoder que servirá para Mahalanobis.
        x = self.transformer(x)

        cls_output = x[:, 0] # Representación latente profunda para OSR
        logits = self.mlp_head(cls_output)

        # Devolvemos (Logits, CLS Token, Attention Weights Placeholder)
        return logits, cls_output, None

# ==============================================================================
# 3. ORQUESTADOR: ESTUDIO DE ABLACIÓN Y ENTRENAMIENTO (FR7, NFR5, FR11)
# ==============================================================================
def train_ablation_study(mode):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f"[*] Acelerador detectado: {device}")

    n_min_candidates = [3, 6, 9, 12, 15, 18]
    epochs_per_ablation = 5 if mode == 'pilot' else GLOBAL_CONFIG['training']['epochs']
    batch_size = 32 if mode == 'pilot' else GLOBAL_CONFIG['training']['batch_size']

    scaler_json = GLOBAL_CONFIG['paths']['configs']['scaler_bounds']
    train_dir = GLOBAL_CONFIG['paths']['output']['train_val']
    ckpt_dir = GLOBAL_CONFIG['paths']['artifacts']['checkpoints']
    os.makedirs(ckpt_dir, exist_ok=True)

    for n_min in n_min_candidates:
        logging.info(f"\n{'='*50}\n[*] INICIANDO ABLACIÓN PARA N_min = {n_min}\n{'='*50}")

        dataset = IDS2018Dataset(train_dir, scaler_json, n_min, mode=mode)
        # FR7: Mitigación de Desbalanceo Matemático (Class Weights)
        total_samples = sum(dataset.class_counts.values())
        class_weights = torch.tensor([total_samples / (len(dataset.class_counts) * (count + 1e-6)) for count in dataset.class_counts.values()], dtype=torch.float32).to(device)

        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, prefetch_factor=None, collate_fn=safe_collate)

        model = ViT_OSR(n_min=n_min).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.AdamW(model.parameters(), lr=GLOBAL_CONFIG['training']['learning_rate'], weight_decay=0.01)
        scaler = torch.amp.GradScaler('cuda') # NFR5: Precisión Mixta

        # FR11: Carga de Checkpoints (Resiliencia)
        start_epoch = 0
        ckpt_path = os.path.join(ckpt_dir, f"vit_nmin_{n_min}_checkpoint.pt")
        if os.path.exists(ckpt_path):
            checkpoint = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(checkpoint['model_state'])
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            scaler.load_state_dict(checkpoint['scaler_state'])
            start_epoch = checkpoint['epoch'] + 1
            logging.info(f"[*] Rescatando entrenamiento desde la Época {start_epoch}")

        if start_epoch >= epochs_per_ablation:
            logging.info(f"[*] Ablación N_min={n_min} ya completada. Saltando.")
            continue

        # Bucle de Entrenamiento
        for epoch in range(start_epoch, epochs_per_ablation):
            model.train()
            running_loss = 0.0
            all_preds, all_labels = [], []

            pbar = tqdm(dataloader, desc=f"Epoca {epoch+1}/{epochs_per_ablation}")
            for inputs, labels in pbar:
                if len(inputs) == 0: continue # Saltar si el lote falló por completo
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad(set_to_none=True)

                # NFR5: AMP
                with torch.amp.autocast('cuda'):
                    logits, _, _ = model(inputs)
                    loss = criterion(logits, labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                pbar.set_postfix(loss=loss.item())

            # Métricas
            epoch_loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0.0
            mcc = matthews_corrcoef(all_labels, all_preds) if len(all_labels) > 0 else 0.0
            logging.info(f"[N_min={n_min}] Época {epoch+1} | Loss: {epoch_loss:.4f} | MCC Train: {mcc:.4f}")

            # FR11: Guardado Atómico Atómico
            tmp_ckpt = ckpt_path + ".tmp"
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scaler_state': scaler.state_dict(),
                'mcc': mcc
            }, tmp_ckpt)
            os.rename(tmp_ckpt, ckpt_path)

train_ablation_study(mode='pilot')

Epoca 2/5: 100%|██████████| 32/32 [00:02<00:00, 14.35it/s, loss=0.00849]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
Epoca 3/5: 100%|██████████| 32/32 [00:01<00:00, 16.69it/s, loss=0.00602]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
Epoca 4/5: 100%|██████████| 32/32 [00:01<00:00, 16.73it/s, loss=0.00476]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.


In [7]:
import os
import torch
import json
import numpy as np

print("=======================================================")
print(" 🕵️ INICIANDO AUDITORÍA FORENSE DEL SANDBOX")
print("=======================================================")

# ==========================================================
# 1. AUDITORÍA DE CHECKPOINTS
# ==========================================================
print("\n[1] VERIFICANDO CHECKPOINTS...")
ckpt_dir = 'checkpoints'
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]

if not ckpts:
    print(" ❌ No se encontraron checkpoints.")
else:
    for ckpt in ckpts:
        try:
            ruta = os.path.join(ckpt_dir, ckpt)
            checkpoint = torch.load(ruta, map_location='cpu', weights_only=False)
            keys = list(checkpoint.keys())
            epoch = checkpoint.get('epoch', 'N/A')
            mcc = checkpoint.get('mcc', 'N/A')
            print(f" ✅ {ckpt} cargado exitosamente.")
            print(f"    -> Época guardada: {epoch} | MCC Final: {mcc}")
            print(f"    -> Componentes atómicos presentes: {keys}")
        except Exception as e:
            print(f" ❌ CORRUPCIÓN EN {ckpt}: {str(e)}")

# ==========================================================
# 2. AUDITORÍA DE LOGS Y TELEMETRÍA
# ==========================================================
print("\n[2] VERIFICANDO LOGS (Errores silenciosos)...")
log_file = 'logs/phase3_ablation.log'

if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        lineas = f.readlines()

    errores = [linea for linea in lineas if "[!]" in linea or "Error" in linea]
    epochs = [linea for linea in lineas if "Época" in linea and "Loss" in linea]

    print(f" 📊 Épocas registradas: {len(epochs)}")
    if errores:
        print(f" ⚠️ Se encontraron {len(errores)} fallos tolerados (safe_collate actuó):")
        for err in errores[:3]: # Mostrar solo los 3 primeros
            print(f"    -> {err.strip()}")
    else:
        print(" ✅ Código 100% limpio. Cero errores de lectura en el DataLoader.")
else:
    print(" ❌ No se encontró archivo de log.")

# ==========================================================
# 3. AUDITORÍA DE ESCALADO Y TENSORES (DATA)
# ==========================================================
print("\n[3] VERIFICANDO LÍMITES MATEMÁTICOS DE ESCALADO...")
scaler_file = 'scaler_bounds.json'

if os.path.exists(scaler_file):
    with open(scaler_file, 'r') as f:
        bounds = json.load(f)

    min_e = bounds['entropy_channel']['min']
    max_e = bounds['entropy_channel']['max']
    min_r = bounds['raw_bytes_channel']['min']
    max_r = bounds['raw_bytes_channel']['max']

    print(" 📏 Fronteras detectadas en la Fase 2:")
    print(f"    -> Entropía: [{min_e}, {max_e}]")
    print(f"    -> Bytes Crudos: [{min_r}, {max_r}]")

    # Prueba lógica
    if min_r == max_r or min_e == max_e:
        print(" ❌ ALERTA CRÍTICA: Min y Max son iguales. Habrá división por cero en el escalado.")
    else:
        print(" ✅ Límites matemáticamente seguros.")
else:
    print(" ❌ No se encontró scaler_bounds.json.")

print("\n=======================================================")
print(" FIN DE LA AUDITORÍA")
print("=======================================================")

 🕵️ INICIANDO AUDITORÍA FORENSE DEL SANDBOX

[1] VERIFICANDO CHECKPOINTS...
 ✅ vit_nmin_18_checkpoint.pt cargado exitosamente.
    -> Época guardada: 4 | MCC Final: 0.0
    -> Componentes atómicos presentes: ['epoch', 'model_state', 'optimizer_state', 'scaler_state', 'mcc']
 ✅ vit_nmin_3_checkpoint.pt cargado exitosamente.
    -> Época guardada: 4 | MCC Final: 0.0
    -> Componentes atómicos presentes: ['epoch', 'model_state', 'optimizer_state', 'scaler_state', 'mcc']
 ✅ vit_nmin_12_checkpoint.pt cargado exitosamente.
    -> Época guardada: 4 | MCC Final: 0.0
    -> Componentes atómicos presentes: ['epoch', 'model_state', 'optimizer_state', 'scaler_state', 'mcc']
 ✅ vit_nmin_9_checkpoint.pt cargado exitosamente.
    -> Época guardada: 4 | MCC Final: 0.0
    -> Componentes atómicos presentes: ['epoch', 'model_state', 'optimizer_state', 'scaler_state', 'mcc']
 ✅ vit_nmin_6_checkpoint.pt cargado exitosamente.
    -> Época guardada: 4 | MCC Final: 0.0
    -> Componentes atómicos presentes:

In [8]:
from google.colab import drive
import os

print("=======================================================")
print(" ☁️ INICIANDO RESPALDO EN GOOGLE DRIVE")
print("=======================================================")

# 1. Montar tu Google Drive (Aparecerá un pop-up pidiendo permisos)
drive.mount('/content/drive')

# 2. Definir y crear la carpeta de destino en tu Drive
# Puedes cambiar "Proyecto_ViT_Backup" por el nombre que prefieras
backup_dir = '/content/drive/MyDrive/Proyecto_ViT_Backup'
os.makedirs(backup_dir, exist_ok=True)
print(f"\n[*] Carpeta destino lista en: {backup_dir}")

# 3. Ejecutar la copia recursiva directamente a nivel de sistema operativo
print("[*] Copiando 'checkpoints/'...")
!cp -r checkpoints/ "{backup_dir}/"

print("[*] Copiando 'data/'...")
!cp -r data/ "{backup_dir}/"

# En caso de que se haya generado algún log, lo copiamos (ignoramos el error si no existe)
print("[*] Copiando 'logs/' (si existe)...")
!cp -r logs/ "{backup_dir}/" 2>/dev/null || true

print("[*] Copiando 'scaler_bounds.json'...")
!cp scaler_bounds.json "{backup_dir}/"

print("\n=======================================================")
print(" [✓] RESPALDO COMPLETADO EXITOSAMENTE")
print("=======================================================")

 ☁️ INICIANDO RESPALDO EN GOOGLE DRIVE
Mounted at /content/drive

[*] Carpeta destino lista en: /content/drive/MyDrive/Proyecto_ViT_Backup
[*] Copiando 'checkpoints/'...
[*] Copiando 'data/'...
[*] Copiando 'logs/' (si existe)...
[*] Copiando 'scaler_bounds.json'...

 [✓] RESPALDO COMPLETADO EXITOSAMENTE
